In [1]:
"""
Solving Ax = b Using LU (or LUP) Decomposition

We want to solve

    Ax = b

where
          [1 2 0]
    A  =  [3 4 4]
          [5 6 3]

and
          [3]
    b  =  [7]
          [8]

--------------------------------------------------
Why don't we solve Ax = b directly?
--------------------------------------------------

In the original system, every equation contains several unknowns:

    x1 + 2x2        = 3
    3x1 + 4x2 + 4x3 = 7
    5x1 + 6x2 + 3x3 = 8

All variables are mixed together.

LU decomposition factors A into

    A = LU

where

    L = lower triangular
    U = upper triangular

Triangular systems are much easier to solve because each row
contains only variables that have already been computed.

The expensive part is computing LU once (O(n^3)).

After that, solving for different b vectors only requires
forward and backward substitution (O(n^2)).

--------------------------------------------------
Step 1: Compute LU
--------------------------------------------------

Start with

          [1 2 0]
    A  =  [3 4 4]
          [5 6 3]

Use row 1 as the first pivot.

Multiplier for row 2:

    l21 = 3

Multiplier for row 3:

    l31 = 5

Perform elimination:

    R2 <- R2 - 3R1

    [3 4 4] - 3[1 2 0]
    [0 -2 4]

    R3 <- R3 - 5R1

    [5 6 3] - 5[1 2 0]
    [0 -4 3]

Matrix becomes

    [1  2  0]
    [0 -2  4]
    [0 -4  3]

Now eliminate below the second pivot.

Multiplier:

    l32 = (-4)/(-2) = 2

Perform

    R3 <- R3 - 2R2

    [0 -4 3] - 2[0 -2 4]
    [0  0 -5]

Therefore

          [1  2  0]
    U  =  [0 -2  4]
          [0  0 -5]

The multipliers form L:

          [1 0 0]
    L  =  [3 1 0]
          [5 2 1]

Verify:

    LU = A

--------------------------------------------------
Step 2: Solve Ly = b
--------------------------------------------------

We solve

          [1 0 0] [y1]   [3]
          [3 1 0] [y2] = [7]
          [5 2 1] [y3]   [8]

Row 1:

    y1 = 3

Row 2:

    3y1 + y2 = 7

    3(3) + y2 = 7

    y2 = -2

Row 3:

    5y1 + 2y2 + y3 = 8

    5(3) + 2(-2) + y3 = 8

    15 - 4 + y3 = 8

    y3 = -3

Therefore

          [ 3]
    y  =  [-2]
          [-3]

This process is called FORWARD SUBSTITUTION.

Why "forward"?

Because we solve

    y1 first,
    then y2,
    then y3.

We move from the top row downward.

--------------------------------------------------
Step 3: Solve Ux = y
--------------------------------------------------

Now solve

          [1  2  0] [x1]   [ 3]
          [0 -2  4] [x2] = [-2]
          [0  0 -5] [x3]   [-3]

Start from the bottom row.

Row 3:

    -5x3 = -3

    x3 = 3/5

Row 2:

    -2x2 + 4x3 = -2

    -2x2 + 4(3/5) = -2

    -2x2 + 12/5 = -2

    -2x2 = -22/5

    x2 = 11/5

Row 1:

    x1 + 2x2 = 3

    x1 + 2(11/5) = 3

    x1 + 22/5 = 15/5

    x1 = -7/5

Therefore

          [-7/5]
    x  =  [11/5]
          [ 3/5]

This process is called BACKWARD SUBSTITUTION.

Why "backward"?

Because we solve

    x3 first,
    then x2,
    then x1.

We move from the bottom row upward.

--------------------------------------------------
Why does LU solving work?
--------------------------------------------------

Since

    A = LU

the original system

    Ax = b

becomes

    LUx = b

Define

    y = Ux

Then

    Ly = b

which is easy to solve by forward substitution.

After finding y, solve

    Ux = y

by backward substitution.

Instead of solving one complicated system, we solve two
simple triangular systems.
"""

'\nSolving Ax = b Using LU (or LUP) Decomposition\n\nWe want to solve\n\n    Ax = b\n\nwhere\n          [1 2 0]\n    A  =  [3 4 4]\n          [5 6 3]\n\nand\n          [3]\n    b  =  [7]\n          [8]\n\n--------------------------------------------------\nWhy don\'t we solve Ax = b directly?\n--------------------------------------------------\n\nIn the original system, every equation contains several unknowns:\n\n    x1 + 2x2        = 3\n    3x1 + 4x2 + 4x3 = 7\n    5x1 + 6x2 + 3x3 = 8\n\nAll variables are mixed together.\n\nLU decomposition factors A into\n\n    A = LU\n\nwhere\n\n    L = lower triangular\n    U = upper triangular\n\nTriangular systems are much easier to solve because each row\ncontains only variables that have already been computed.\n\nThe expensive part is computing LU once (O(n^3)).\n\nAfter that, solving for different b vectors only requires\nforward and backward substitution (O(n^2)).\n\n--------------------------------------------------\nStep 1: Compute LU\n---

In [2]:
"""
A useful intuition is that plain LU and LUP are solving the same problem from different starting points.
Plain LU starts with row 1 as the first pivot. LUP says, "Before eliminating, let's move the best pivot to the top."
The permutation matrix P keeps track of those swaps so the mathematics stays equivalent. The final x is identical either way.
"""

'\nA useful intuition is that plain LU and LUP are solving the same problem from different starting points.\nPlain LU starts with row 1 as the first pivot. LUP says, "Before eliminating, let\'s move the best pivot to the top."\nThe permutation matrix P keeps track of those swaps so the mathematics stays equivalent. The final x is identical either way.\n'

In [ ]:
"""
There is not a unique LU decomposition.

One approach is to perform Gaussian elimination directly on A.

Another approach is to first swap rows so that the largest
available pivot is used. This is called partial pivoting.

With pivoting, we compute

    PA = LU

instead of

    A = LU

where

P = permutation matrix
L = lower triangular matrix
U = upper triangular matrix

Both methods solve the same system Ax = b and produce the
same final solution x.

The pivoted method is usually preferred because it is more
numerically stable and avoids dividing by very small numbers.

--------------------------------------------------
Step 1: Build the permutation matrix P
--------------------------------------------------

Look at the first column of A:

    [1]
    [3]
    [5]

The largest value is 5, which is in row 3.

Swap row 1 and row 3.

The permutation matrix is

          [0 0 1]
P =       [1 0 0]
          [0 1 0]

Multiplying P by A rearranges the rows:

           [5 6 3]
PA =       [1 2 0]
           [3 4 4]

--------------------------------------------------
Step 2: Eliminate below the first pivot
--------------------------------------------------

Pivot:

    5

--------------------------------------------------
Row 2 multiplier
--------------------------------------------------

The entry below the pivot is 1.

To eliminate it:

    multiplier = 1/5 = 0.2

Store:

    l21 = 0.2

Perform

    R2 <- R2 - 0.2 R1

    [1 2 0]
  - 0.2[5 6 3]

    [1 2 0]
  - [1 1.2 0.6]

    [0 0.8 -0.6]

--------------------------------------------------
Row 3 multiplier
--------------------------------------------------

The entry below the pivot is 3.

To eliminate it:

    multiplier = 3/5 = 0.6

Store:

    l31 = 0.6

Perform

    R3 <- R3 - 0.6 R1

    [3 4 4]
  - 0.6[5 6 3]

    [3 4 4]
  - [3 3.6 1.8]

    [0 0.4 2.2]

Current matrix:

    [5 6   3  ]
    [0 0.8 -0.6]
    [0 0.4 2.2]

--------------------------------------------------
Step 3: Eliminate below the second pivot
--------------------------------------------------

Pivot:

    0.8

Entry to eliminate:

    0.4

Multiplier:

    0.4 / 0.8 = 0.5

Store:

    l32 = 0.5

Perform

    R3 <- R3 - 0.5 R2

    [0 0.4 2.2]
  - 0.5[0 0.8 -0.6]

    [0 0.4 2.2]
  - [0 0.4 -0.3]

    [0 0 2.5]

--------------------------------------------------
Step 4: Construct L and U
--------------------------------------------------

The multipliers become the entries of L:

          [1   0   0]
L =       [0.2 1   0]
          [0.6 0.5 1]

The final matrix is U:

          [5 6   3  ]
U =       [0 0.8 -0.6]
          [0 0   2.5]

Verify:

    PA = LU

--------------------------------------------------
Step 5: Solve Ly = Pb
--------------------------------------------------

First compute Pb:

          [0 0 1]   [3]
Pb =      [1 0 0] * [7]
          [0 1 0]   [8]

          [8]
Pb =      [3]
          [7]

Now solve

          [1   0   0 ] [y1]   [8]
          [0.2 1   0 ] [y2] = [3]
          [0.6 0.5 1 ] [y3]   [7]

Row 1:

    y1 = 8

Row 2:

    0.2(8) + y2 = 3

    1.6 + y2 = 3

    y2 = 1.4

Row 3:

    0.6(8) + 0.5(1.4) + y3 = 7

    4.8 + 0.7 + y3 = 7

    y3 = 1.5

Therefore

          [8  ]
y =       [1.4]
          [1.5]

This is called FORWARD SUBSTITUTION.

--------------------------------------------------
Step 6: Solve Ux = y
--------------------------------------------------

          [5 6   3  ] [x1]   [8  ]
          [0 0.8 -0.6] [x2] = [1.4]
          [0 0   2.5] [x3]   [1.5]

Start from the bottom.

Row 3:

    2.5x3 = 1.5

    x3 = 0.6

Row 2:

    0.8x2 - 0.6(0.6) = 1.4

    0.8x2 - 0.36 = 1.4

    0.8x2 = 1.76

    x2 = 2.2

Row 1:

    5x1 + 6(2.2) + 3(0.6) = 8

    5x1 + 13.2 + 1.8 = 8

    5x1 = -7

    x1 = -1.4

Therefore

          [-1.4]
x =       [ 2.2]
          [ 0.6]

or

          [-7/5]
x =       [11/5]
          [ 3/5]

--------------------------------------------------
Why does this method still work?
--------------------------------------------------

Originally we want to solve

    Ax = b

The permutation matrix only rearranges rows.

Multiplying both sides by P gives

    PAx = Pb

Since

    PA = LU

we get

    LUx = Pb

Let

    y = Ux

Then

    Ly = Pb

Solve for y using forward substitution.

Then solve

    Ux = y

using backward substitution.

The row swaps do not change the solution x.
They only change the order of the equations.

The permutation matrix records those row swaps so that
the algebra remains correct.
"""

In [3]:
def lup_solve(L, U, pi, b):
    n = len(b)

    y = [0.0] * n
    x = [0.0] * n

    # Forward substitution: Ly = Pb
    for i in range(n):
        y[i] = b[pi[i]]

        for j in range(i):
            y[i] -= L[i][j] * y[j]

    # Backward substitution: Ux = y
    for i in range(n - 1, -1, -1):
        x[i] = y[i]

        for j in range(i + 1, n):
            x[i] -= U[i][j] * x[j]

        x[i] /= U[i][i]

    return x

In [4]:
"""
The Schur complement is not a different answer and not a different decomposition.
It's a different way of looking at the same elimination process.

In ordinary Gaussian elimination, you think:
"I'm eliminating one row at a time."

With the Schur complement viewpoint, you think:
"I'm eliminating one block of variables and updating the remaining block."

The numbers are identical, only the perspective changes.
"""

'\nThe Schur complement is not a different answer and not a different decomposition.\nIt\'s a different way of looking at the same elimination process.\n\nIn ordinary Gaussian elimination, you think:\n"I\'m eliminating one row at a time."\n\nWith the Schur complement viewpoint, you think:\n"I\'m eliminating one block of variables and updating the remaining block."\n\nThe numbers are identical, only the perspective changes.\n'

In [5]:
"""
# Schur Complement Intuition

The Schur complement is not a different answer.

It is a different way of describing the same elimination
that occurs during LU decomposition.

--------------------------------------------------
Ordinary Gaussian Elimination View
--------------------------------------------------
Suppose
          [5 6 3]
PA =      [1 2 0]
          [3 4 4]

We use the pivot 5.

Multiplier for row 2:
    1/5 = 0.2

Multiplier for row 3:
    3/5 = 0.6

Perform row operations:
    R2 <- R2 - 0.2 R1
    R3 <- R3 - 0.6 R1

Result:
    [5 6   3  ]
    [0 0.8 -0.6]
    [0 0.4  2.2]

This is the usual Gaussian elimination viewpoint.

--------------------------------------------------
Schur Complement View
--------------------------------------------------

Instead of focusing on rows, split the matrix into blocks.

          [ A11   A12 ]
PA =      [ A21   A22 ]

where

A11 = [5]
A12 = [6 3]
A21 = [1]
      [3]
A22 = [2 0]
      [4 4]

The Schur complement is
    S = A22 - A21 A11^(-1) A12

--------------------------------------------------
Compute the pieces
--------------------------------------------------

First:
    A11^(-1) = 1/5

Next:

          [1]
A21A11^(-1) =
          [3] * (1/5)

          [0.2]
        = [0.6]

Notice these are exactly the first-column multipliers
that appear in L.

Now multiply
          [0.2]
          [0.6]

by

    [6 3]

Result:
    [1.2 0.6]
    [3.6 1.8]

Now subtract from A22:

    [2 0]     [1.2 0.6]
    [4 4]  -  [3.6 1.8]

    [0.8 -0.6]
    [0.4  2.2]

--------------------------------------------------
The Key Observation
--------------------------------------------------

The resulting matrix

    [0.8 -0.6]
    [0.4  2.2]

is exactly the lower-right block obtained after the
first elimination step.

So the Schur complement is simply:

    "the remaining matrix after eliminating the first
     pivot block."

--------------------------------------------------
Why Use Blocks?
--------------------------------------------------

For a small 3x3 matrix, the row-by-row method is easier.

For a huge matrix, it is often easier to think in blocks.

Instead of eliminating one row at a time:

    eliminate a block

then

    update the remaining block

using a Schur complement.

--------------------------------------------------
Relationship to LU
--------------------------------------------------

Gaussian elimination says:

    eliminate rows

Schur complement says:

    update the remaining block

Both describe exactly the same computation.

The values in L, U, and the final solution x are
identical.

Only the viewpoint changes.

--------------------------------------------------
Simple Memory Trick
--------------------------------------------------

LU viewpoint:

    "What row operations do I perform?"

Schur complement viewpoint:

    "After eliminating a block, what smaller matrix
     remains to be solved?"

The smaller remaining matrix is the Schur complement.
"""

'\n# Schur Complement Intuition\n\nThe Schur complement is not a different answer.\n\nIt is a different way of describing the same elimination\nthat occurs during LU decomposition.\n\n--------------------------------------------------\nOrdinary Gaussian Elimination View\n--------------------------------------------------\nSuppose\n          [5 6 3]\nPA =      [1 2 0]\n          [3 4 4]\n\nWe use the pivot 5.\n\nMultiplier for row 2:\n    1/5 = 0.2\n\nMultiplier for row 3:\n    3/5 = 0.6\n\nPerform row operations:\n    R2 <- R2 - 0.2 R1\n    R3 <- R3 - 0.6 R1\n\nResult:\n    [5 6   3  ]\n    [0 0.8 -0.6]\n    [0 0.4  2.2]\n\nThis is the usual Gaussian elimination viewpoint.\n\n--------------------------------------------------\nSchur Complement View\n--------------------------------------------------\n\nInstead of focusing on rows, split the matrix into blocks.\n\n          [ A11   A12 ]\nPA =      [ A21   A22 ]\n\nwhere\n\nA11 = [5]\nA12 = [6 3]\nA21 = [1]\n      [3]\nA22 = [2 0]\n    

In [6]:
"""
Extract one row of U.
Extract one column of L.
Replace the remaining submatrix with its Schur complement.
Repeat on the smaller submatrix.
"""
def lu_decomposition(A):
    n = len(A)

    # Create L and U
    L = [[0.0] * n for _ in range(n)]
    U = [[0.0] * n for _ in range(n)]

    # Initialize L diagonal to 1
    for i in range(n):
        L[i][i] = 1.0

    # Make a working copy so we don't destroy the original
    A = [row[:] for row in A]

    for k in range(n):
        # Pivot
        U[k][k] = A[k][k]

        # Compute kth column of L and kth row of U
        for i in range(k + 1, n):
            L[i][k] = A[i][k] / A[k][k]
            U[k][i] = A[k][i]

        # Compute Schur complement
        for i in range(k + 1, n):
            for j in range(k + 1, n):
                A[i][j] = A[i][j] - L[i][k] * U[k][j]

    return L, U

In [7]:
A = [
    [5, 6, 3],
    [1, 2, 0],
    [3, 4, 4]
]

L, U = lu_decomposition(A)

print("L:")
for row in L:
    print(row)

print("\nU:")
for row in U:
    print(row)

L:
[1.0, 0.0, 0.0]
[0.2, 1.0, 0.0]
[0.6, 0.5000000000000006, 1.0]

U:
[5, 6, 3]
[0.0, 0.7999999999999998, -0.6000000000000001]
[0.0, 0.0, 2.5000000000000004]


In [ ]:
"""
A modern LU algorithm really looks like:

Choose a pivot (possibly swap rows).
Form the multipliers for L.
Compute the Schur complement.
Repeat on the smaller Schur complement.

So the workflow is:

Choose pivot
      ↓
Compute multipliers
      ↓
Update Schur complement
      ↓
Repeat

The Schur complement is the updated remaining matrix.
Pivoting is the strategy for choosing a safe pivot before the update.

LU decomposition = factor the matrix.
Schur complement = the remaining subproblem after elimination.
Pivoting (PA or QA) = rearrange rows/columns so elimination is safe and numerically stable.

Pivoting (Q or P) chooses a safe pivot.
The multipliers form L.
The remaining block is the Schur complement.
LU decomposition is just repeating this block factorization recursively.
"""